In [1]:
include("code.jl")

Code correctly loaded.


## Permanent approximation  

Computing the permanent is known to be a hard task, thus using the best known algorithm (Ryser algorithm) is not the best choice. We can use approximate algorithm like Gurvits, which are able to approximate $\operatorname{perm}(A)$ up to erro $\epsilon \lVert A\rVert^{n}$ in $\mathcal{O}(n^{2}/\epsilon^{2})$. Notice that the error is exponential in the largest eigenvalue. Nevertheless to compute the characteristic function we just need to compute permanents of unitary matrix, thus the largest eigenvalue will always have modulus $1$, making the algorithm able to give us additive error on the characteristic function.

In [2]:
n=10
k=2
U=RandHaar(n).U
A=U'*diagm([ones(Int, k); zeros(Int, n - k)])*U

Perm=ryser(A)
Estimated_Perm=estimate_permanent(A)
println("Permanent:           ", Perm)
println("Estimated Permanent: ", Estimated_Perm)
println("Absolute Error:      ", abs(Perm - Estimated_Perm))

Permanent:           0.00021183974488543457 - 1.665937617778735e-19im
Estimated Permanent: 0.00020504885807148585 + 7.935462012614025e-6im
Absolute Error:      1.0444505784071621e-5


To compute the output probability distribution of a boson sampling machine, we can compute 
\begin{equation}
    \chi(\phi_{1},...,\phi_{m}|U)=\langle\Psi_{in}|U^{\dagger}e^{i\sum_{j}\phi_{j}\hat{n}_{j}}U|\Psi_{in}\rangle=\operatorname{perm}\left(U^{\dagger}\Phi U\right)
\end{equation}
with $\Phi=\operatorname{diag}(\phi_{1},...,\phi_{m})$. In the case we consider partial distinguishable particles, we can take care of it by adding the distinguishability matrix $S$ as follows
\begin{equation}
    \chi(\phi_{1},...,\phi_{m}|U,S)=\operatorname{perm}\left(U^{\dagger}\Phi U\odot S\right)
\end{equation}
which reduces to the above in the case on indistinguishable particles, since it correspond to $S_{ij}=1\forall i,j$.

In the following we are interested in the two mode correlators, which can be expressed as
\begin{align}
    \langle \hat{n}_{i}\hat{n}_{j}\rangle &= \delta_{ij} \sum_{k=1}^n |U_{ik}|^2 + \sum_{k \neq l} \left( |U_{ik}|^2 |U_{jl}|^2 + |S_{kl}|^2 U_{ik}^* U_{il} U_{jl}^* U_{jk} \right) \quad .
\end{align}
Notice that if we are interested to a system with internal degrees of freedom which are mixed states we can just substitute $\operatorname{Tr}[\rho_{i}\rho_{j}]$ to the term $ |S_{ij}|^2$.

## Obtaining the Characteristic function 

We consider two copies of our system which evolves through a linear unitary $U$ and then we apply virtual distillation. The overall matrix to diagonalize to compute the characteristic function is
\begin{equation}
    U_{tot}=e^{i\sum_{j}\phi_{j}\hat{n}_{j,1}}\hat{S}_{2}(U\oplus U)
\end{equation}

In [3]:
m=2

U=RandHaar(m).U
ϕ=rand(m)

V=U_tot(U,ϕ)

Perm=ryser(V)
Estimated_Perm=CF(ϕ,U,1;Samples=10^6)
println("CF (Ryser)    : ", Perm)
println("CF (Gurvits)  : ", Estimated_Perm)
println("Absolute Error: ", abs(Perm - Estimated_Perm))

CF (Ryser)    : 0.8858244594231228 + 0.4245451645297138im
CF (Gurvits)  : 0.8858123321731437 + 0.42457046833711337im
Absolute Error: 2.8059808641017865e-5


## Expectation value from the derivatives   

Let us consider now thermal states.

In [58]:
m=6
U=Fourier(m).U;
abs2.(U)

6×6 Matrix{Float64}:
 0.166667  0.166667  0.166667  0.166667  0.166667  0.166667
 0.166667  0.166667  0.166667  0.166667  0.166667  0.166667
 0.166667  0.166667  0.166667  0.166667  0.166667  0.166667
 0.166667  0.166667  0.166667  0.166667  0.166667  0.166667
 0.166667  0.166667  0.166667  0.166667  0.166667  0.166667
 0.166667  0.166667  0.166667  0.166667  0.166667  0.166667

In [ ]:
β=2*atanh.(collect(LinRange(0.65,0.99,15)))
Number_of_samples=5*10^6
δ= 0.005


y_n=[compute_density_correlation(1,2, U, (1-sqrt(tanh(x/2)))*Matrix(I,m,m)+sqrt(tanh(x/2))*ones(m,m)) for x in β]
y_d=[calc_n1n2_thermal(δ,U,x;num_samples=Number_of_samples) for x in β];

In [ ]:
# Fit a quadratic polynomial to the data
x = tanh.(β ./ 2)
y = y_d

# 2. Construct the design matrix A = [1  x  x^2]
A = [ones(length(x)) x x.^2 ]

# 3. Solve for the coefficients [a_0, a_1, a_2] using least squares
coeffs = A \ y
y_fitted = A * coeffs;

In [ ]:
Ideal=compute_density_correlation(1,2, U,ones(m,m))*ones(length(β))
y_m=[compute_density_correlation(1,2, U, (1-sqrt(tanh(x)))*Matrix(I,m,m)+sqrt(tanh(x))*ones(m,m)) for x in β];

In [ ]:
norm(y_fitted-y_m)

In [ ]:

# Pre-calculate data
x_data = tanh.(β ./ 2)
ideal_y = Ideal .* ones(length(β))

# Apply global styles mirroring the PDF
default(
    fontfamily="serif",
    framestyle=:box,                 
    grid=false,                      
    tick_direction=:in,              
    legend=:topright,      
    foreground_color_legend=nothing, 
    dpi=600,
    markerstrokecolor=:auto          # <-- This forces the stroke to match the line/fill color
)

mark=7

# 1. Use `scatter` instead of `plot` to plot ONLY markers (no lines)
scatter(x_data, y_n, label="Noisy", xlabel=L"\mathrm{Tr}[\rho^2]", ylabel=L"\langle n_1 n_2 \rangle", marker=:+, markersize=mark,color="#0F1F7A")
plot!(x_data, y_m, label="Half temperature",color="#0DA0A3",linestyle=:dash)
scatter!(x_data, y_d, label="M=2", marker=:+, markersize=mark,color="#D60F0F")

# 2. Keep `plot!` for the Ideal line since you want a dashed line, not markers
plot!(x_data, ideal_y, label="Ideal", linestyle=:dash, color="#939393", linewidth=1.5,xlimits=(0.65,1))
#savefig("Correlator_Distillation.pdf")